# Notebook 1: Exploratory Data Analysis (EDA)
## Sentinel Risk Engine

This notebook performs an initial exploratory data analysis of the credit card fraud dataset (`fraudTrain.csv` and `fraudTest.csv`).

### Objectives:
1. Understand dataset dimensions, schema, and missing values.
2. Analyze target class imbalance (`is_fraud`).
3. Explore transaction amount (`amt`) distributions for fraudulent vs. non-fraudulent transactions.
4. Analyze categorical patterns (e.g., transaction category, merchant).
5. Compute geospatial distance features between cardholder and merchant.
6. Investigate temporal patterns (hour of day, day of week).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

DATA_PATH = os.path.join("..", "data", "raw", "fraudTrain.csv")
print(f"Data path: {DATA_PATH}")

## 1. Load Data & Basic Summary

In [ ]:
# Load sample or full dataset
df = pd.read_csv(DATA_PATH, index_col=0)
print(f"Dataset Shape: {df.shape}")
print("\n--- Columns & Data Types ---")
print(df.dtypes)
df.head()

## 2. Target Variable Analysis (Class Imbalance)

In [ ]:
fraud_counts = df['is_fraud'].value_counts()
fraud_pct = df['is_fraud'].value_counts(normalize=True) * 100

print("Class Counts:")
print(fraud_counts)
print("\nClass Percentages:")
print(fraud_pct)

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='is_fraud', palette='viridis', ax=ax)
ax.set_title('Target Class Distribution (0: Non-Fraud, 1: Fraud)')
ax.set_yscale('log')
plt.show()

## 3. Transaction Amount Analysis

In [ ]:
print("--- Summary Statistics for Transaction Amount (amt) ---")
print(df.groupby('is_fraud')['amt'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='is_fraud', y='amt', palette='Set2', ax=axes[0])
axes[0].set_title('Transaction Amount Boxplot by Class')
axes[0].set_yscale('log')

sns.kdeplot(data=df, x='amt', hue='is_fraud', common_norm=False, log_scale=True, palette='Set1', ax=axes[1])
axes[1].set_title('Transaction Amount Density Distribution (Log Scale)')
plt.tight_layout()
plt.show()

## 4. Category-Wise Fraud Rate

In [ ]:
cat_summary = df.groupby('category').agg(
    total_tx=('is_fraud', 'count'),
    fraud_tx=('is_fraud', 'sum'),
    fraud_rate=('is_fraud', 'mean')
).sort_values(by='fraud_rate', ascending=False)

print(cat_summary)

plt.figure(figsize=(12, 5))
sns.barplot(x=cat_summary.index, y=cat_summary['fraud_rate'] * 100, palette='magma')
plt.xticks(rotation=45, ha='right')
plt.title('Fraud Rate by Transaction Category (%)')
plt.ylabel('Fraud Rate (%)')
plt.tight_layout()
plt.show()

## 5. Geospatial Distance Feature Engineering

In [ ]:
def haversine_np(lat1, lon1, lat2, lon2):
    """Calculate geodesic distance between two points in km."""
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6367 * c
    return km

df['distance_km'] = haversine_np(df['lat'], df['long'], df['merch_lat'], df['merch_long'])
print(df.groupby('is_fraud')['distance_km'].describe())

plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x='is_fraud', y='distance_km', palette='crest')
plt.title('Cardholder to Merchant Distance (km) by Class')
plt.show()

## 6. Key Insights Summary

- **Class Imbalance**: Fraud instances account for a very small fraction (<1%) of overall transactions.
- **Transaction Amounts**: Fraudulent transactions tend to have higher mean amounts and distinct multi-modal distributions.
- **Categories**: Categories such as `grocery_pos`, `shopping_net`, and `misc_net` exhibit significantly higher fraud rates.
- **Geospatial Distance**: Merchant distance shows variation that provides signal for fraud risk scoring.